# Lab 3.1: CLAUDE.md Hierarchy & Modular Organization

**What you'll build:** A configuration system that demonstrates CLAUDE.md hierarchy levels, @import directives, and the /memory command -- and learn why putting team rules in the wrong level breaks collaboration.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- rules in user-level CLAUDE.md break team sharing | 5 min |
| 2 | The Right Way -- proper hierarchy with project and directory levels | 5 min |
| 3 | Your Turn -- design a CLAUDE.md hierarchy for a real project | 10 min |
| 4 | Stress Test -- validate your hierarchy handles edge cases | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are configuring Claude Code for a team project. The project is a monorepo with:
- A Python backend (`/backend`)
- A TypeScript frontend (`/frontend`)
- Shared database migrations (`/database`)
- Infrastructure configs (`/infra`)

Each area has different conventions, but the whole team shares coding standards.

We'll simulate CLAUDE.md configurations as Python dictionaries and use Claude to evaluate whether a given configuration would work correctly for different scenarios.

In [ ]:
# Simulating CLAUDE.md hierarchy as data structures
# In real Claude Code, these would be actual .md files at specific paths

# A team lead's configuration -- ALL in user-level (the wrong way)
WRONG_CONFIG = {
    "user_level": {  # ~/.claude/CLAUDE.md
        "path": "~/.claude/CLAUDE.md",
        "shared_via_git": False,
        "content": """
# Team Standards
- Use 4-space indentation
- All functions must have type hints
- Use pytest for testing, name tests: test_<behavior>_<condition>_<expected>
- API responses must use camelCase keys
- Database migrations must have rollback scripts
- Frontend components use PascalCase filenames
- Always write docstrings for public functions
- Use PostgreSQL, not MySQL
"""
    },
    "project_level": None,  # No .claude/CLAUDE.md or root CLAUDE.md
    "directory_level": {}   # No subdirectory CLAUDE.md files
}

# Team members who would need these standards
TEAM = [
    {"name": "Alice", "role": "Team Lead", "has_user_config": True},
    {"name": "Bob", "role": "Backend Dev", "has_user_config": False},
    {"name": "Carol", "role": "Frontend Dev", "has_user_config": False},
    {"name": "Dave", "role": "New Hire", "has_user_config": False},
]

print("=== WRONG CONFIGURATION ===")
print(f"Rules location: {WRONG_CONFIG['user_level']['path']}")
print(f"Shared via git: {WRONG_CONFIG['user_level']['shared_via_git']}")
print()
print("Team members who see these rules:")
for member in TEAM:
    sees_rules = member['has_user_config']
    status = 'YES' if sees_rules else 'NO -- rules are in user-level only'
    print(f"  {member['name']} ({member['role']}): {status}")

---

## Phase 1: The Wrong Approach

Let's ask Claude to evaluate this configuration and explain why it fails for team collaboration.

In [ ]:
EVAL_PROMPT = f"""You are evaluating a Claude Code CLAUDE.md configuration for a 4-person team.

The team lead put ALL team standards in ~/.claude/CLAUDE.md (user-level).
There is NO project-level CLAUDE.md and NO directory-level CLAUDE.md files.

Configuration:
{json.dumps(WRONG_CONFIG, indent=2)}

Team:
{json.dumps(TEAM, indent=2)}

Analyze this configuration and return a JSON object with:
- "problems": list of specific problems with this setup
- "affected_members": list of team member names who are affected
- "risk_level": "high", "medium", or "low"
- "root_cause": one sentence explaining the fundamental issue

Return ONLY the JSON object, no other text."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": EVAL_PROMPT}]
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
analysis = json.loads(raw)

print("=== CONFIGURATION ANALYSIS ===")
print(f"Risk Level: {analysis['risk_level'].upper()}")
print(f"Root Cause: {analysis['root_cause']}")
print(f"\nAffected members: {', '.join(analysis['affected_members'])}")
print(f"\nProblems:")
for i, problem in enumerate(analysis['problems'], 1):
    print(f"  {i}. {problem}")

### Why did that fail?

- **User-level CLAUDE.md is not version-controlled.** Only the person who created it sees those rules.
- **New team members get zero guidance.** Dave (new hire) has no CLAUDE.md at all -- Claude Code gives him default behavior.
- **No directory-specific rules.** Backend Python conventions and frontend TypeScript conventions are mixed in one file, applied everywhere.
- **No modularity.** A single 500-line file becomes unmaintainable as the project grows.

---

## Phase 2: The Right Approach

Now let's build a proper CLAUDE.md hierarchy with the three levels working together.

In [ ]:
# The right configuration -- proper hierarchy
RIGHT_CONFIG = {
    "user_level": {  # ~/.claude/CLAUDE.md -- personal only
        "path": "~/.claude/CLAUDE.md",
        "shared_via_git": False,
        "content": """
# Personal Preferences
- I prefer verbose explanations in code comments
- Show me git diff before committing
"""
    },
    "project_level": {  # .claude/CLAUDE.md -- team-wide
        "path": ".claude/CLAUDE.md",
        "shared_via_git": True,
        "content": """
# Project Standards
@import standards/coding.md
@import standards/testing.md
@import standards/architecture.md

## Quick Reference
- Database: PostgreSQL (not MySQL)
- Default model: claude-sonnet-4-20250514
- All functions must have type hints
- Use 4-space indentation everywhere
"""
    },
    "directory_level": {
        "backend/CLAUDE.md": {
            "path": "backend/CLAUDE.md",
            "shared_via_git": True,
            "content": """
# Backend (Python)
- Use FastAPI for endpoints
- SQLAlchemy for ORM
- API responses: camelCase keys (serializer handles conversion)
- Always write docstrings for public functions
"""
        },
        "frontend/CLAUDE.md": {
            "path": "frontend/CLAUDE.md",
            "shared_via_git": True,
            "content": """
# Frontend (TypeScript)
- React with TypeScript strict mode
- Components use PascalCase filenames
- State management: Zustand
- Styles: Tailwind CSS, no inline styles
"""
        },
        "database/CLAUDE.md": {
            "path": "database/CLAUDE.md",
            "shared_via_git": True,
            "content": """
# Database Migrations
- Every migration must have a rollback script
- Use explicit column types (no implicit defaults)
- Name format: YYYYMMDD_HHMMSS_description.sql
"""
        }
    }
}

print("=== CORRECT CONFIGURATION ===")
print("\nUser level (personal, NOT shared):")
print(f"  {RIGHT_CONFIG['user_level']['path']}")
print(f"  Content: personal preferences only")

print(f"\nProject level (team-wide, shared via git):")
print(f"  {RIGHT_CONFIG['project_level']['path']}")
print(f"  Content: universal standards + @import for modularity")

print(f"\nDirectory level (scoped, shared via git):")
for name, config in RIGHT_CONFIG['directory_level'].items():
    print(f"  {config['path']}: {config['content'].strip().split(chr(10))[0]}")

print("\nTeam members who see these rules:")
for member in TEAM:
    print(f"  {member['name']} ({member['role']}): ALL project + directory rules (via git)")

In [ ]:
# Let's ask Claude to evaluate the right configuration
EVAL_RIGHT_PROMPT = f"""You are evaluating a Claude Code CLAUDE.md configuration for a 4-person team.

Configuration uses all three hierarchy levels:
{json.dumps(RIGHT_CONFIG, indent=2)}

Evaluate this configuration and return a JSON object with:
- "strengths": list of things this configuration does well
- "team_coverage": percentage of team members who see the shared rules
- "modularity_score": 1-10 rating of how well-organized this is
- "hierarchy_usage": description of how each level is properly used

Return ONLY the JSON object, no other text."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": EVAL_RIGHT_PROMPT}]
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
right_analysis = json.loads(raw)

print("=== RIGHT CONFIGURATION ANALYSIS ===")
print(f"Team Coverage: {right_analysis['team_coverage']}")
print(f"Modularity Score: {right_analysis['modularity_score']}/10")
print(f"\nStrengths:")
for i, s in enumerate(right_analysis['strengths'], 1):
    print(f"  {i}. {s}")
print(f"\nHierarchy Usage: {right_analysis['hierarchy_usage']}")

### Side-by-Side Comparison

| Aspect | Wrong Config | Right Config |
|--------|-------------|---------------|
| **Team coverage** | 1/4 members (25%) | 4/4 members (100%) |
| **Version controlled** | No | Yes (project + directory) |
| **Modularity** | Single monolithic file | @import + directory-level scoping |
| **Backend-specific rules** | Mixed with everything | Scoped to backend/CLAUDE.md |
| **Frontend-specific rules** | Mixed with everything | Scoped to frontend/CLAUDE.md |
| **Personal preferences** | Mixed with team rules | Separate in user-level |

---

## Phase 3: Your Turn

Design a CLAUDE.md hierarchy for a new project. The project is an e-commerce platform with:
- `/api` -- Python FastAPI backend
- `/web` -- Next.js frontend
- `/mobile` -- React Native mobile app
- `/shared` -- Shared TypeScript types and utilities
- `/infra` -- Terraform infrastructure configs

**Requirements:**
1. All team members must see coding standards
2. API layer must use specific error handling patterns
3. Frontend and mobile share React conventions but differ on styling
4. Infrastructure files have strict change management rules
5. Personal preferences (editor style, verbosity) stay personal

In [ ]:
# =============================================================
# TODO: Design your CLAUDE.md hierarchy
# =============================================================
#
# Fill in each level with appropriate content.
# Think about:
# - What belongs at each level?
# - What should be in @import files vs inline?
# - What directory CLAUDE.md files are needed?

YOUR_CONFIG = {
    "user_level": {
        "path": "~/.claude/CLAUDE.md",
        "shared_via_git": False,
        "content": """
# TODO: Personal preferences only
"""
    },
    "project_level": {
        "path": ".claude/CLAUDE.md",
        "shared_via_git": True,
        "content": """
# TODO: Team-wide standards (consider using @import)
"""
    },
    "directory_level": {
        # TODO: Add directory CLAUDE.md files for each area
        # Example:
        # "api/CLAUDE.md": {
        #     "path": "api/CLAUDE.md",
        #     "shared_via_git": True,
        #     "content": "..."
        # }
    }
}

print("Your configuration:")
print(json.dumps(YOUR_CONFIG, indent=2))

In [ ]:
# =============================================================
# Checker: validates your hierarchy design
# =============================================================

def check_hierarchy(config):
    errors = []
    warnings = []
    
    # Check 1: User level should NOT have team rules
    user_content = config.get('user_level', {}).get('content', '').lower()
    team_keywords = ['team standard', 'coding standard', 'all functions', 'type hint',
                     'indentation', 'api response', 'database', 'migration']
    team_rules_in_user = [kw for kw in team_keywords if kw in user_content]
    if team_rules_in_user:
        errors.append(f"Team rules found in user-level: {team_rules_in_user}. "
                      f"Move these to project-level.")
    
    # Check 2: Project level must exist and be shared
    project = config.get('project_level')
    if not project or not project.get('content', '').strip() or project['content'].strip() == '# TODO: Team-wide standards (consider using @import)':
        errors.append("Project-level CLAUDE.md is empty or missing. "
                      "Team standards must be here.")
    elif not project.get('shared_via_git', False):
        errors.append("Project-level must be shared via git.")
    
    # Check 3: Should have directory-level files
    dirs = config.get('directory_level', {})
    expected_dirs = ['api', 'web', 'mobile', 'infra']
    found_dirs = []
    for dir_path in dirs:
        for expected in expected_dirs:
            if expected in dir_path.lower():
                found_dirs.append(expected)
    
    missing = [d for d in expected_dirs if d not in found_dirs]
    if missing:
        warnings.append(f"Missing directory CLAUDE.md for: {missing}. "
                        f"Consider adding scoped rules.")
    
    # Check 4: Project level should use @import for modularity
    if project and '@import' not in project.get('content', ''):
        warnings.append("Consider using @import in project-level CLAUDE.md "
                        "for modularity as the config grows.")
    
    # Results
    print("=== HIERARCHY VALIDATION ===")
    if not errors and not warnings:
        print("PASSED -- Well-designed hierarchy!")
    else:
        if errors:
            print(f"ERRORS ({len(errors)}):")
            for e in errors:
                print(f"  [ERROR] {e}")
        if warnings:
            print(f"WARNINGS ({len(warnings)}):")
            for w in warnings:
                print(f"  [WARN] {w}")
        if not errors:
            print("\nNo errors -- warnings are suggestions for improvement.")
        else:
            print("\nFix the errors and try again.")
    
    return len(errors) == 0

check_hierarchy(YOUR_CONFIG)

---

## Phase 4: Stress Test

Let's verify your hierarchy handles real-world scenarios correctly by asking Claude to evaluate what instructions it would receive for different file edits.

In [ ]:
# Stress test: ask Claude what instructions apply to different files
SCENARIOS = [
    {"file": "api/routes/users.py", "expected_levels": ["project", "directory"]},
    {"file": "web/components/Header.tsx", "expected_levels": ["project", "directory"]},
    {"file": "infra/main.tf", "expected_levels": ["project", "directory"]},
    {"file": "README.md", "expected_levels": ["project"]},
]

STRESS_PROMPT = f"""Given this CLAUDE.md hierarchy:
{json.dumps(YOUR_CONFIG, indent=2)}

For each file below, determine which CLAUDE.md levels would be loaded and merged:
{json.dumps(SCENARIOS, indent=2)}

Return a JSON array where each item has:
- "file": the filename
- "loaded_levels": which CLAUDE.md levels apply ("user", "project", "directory")
- "key_instructions": 2-3 most relevant instructions from the merged config
- "correct_hierarchy": true if the hierarchy provides appropriate guidance for this file

Return ONLY the JSON array."""

response = client.messages.create(
    model=MODEL,
    max_tokens=2000,
    messages=[{"role": "user", "content": STRESS_PROMPT}]
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    stress_results = json.loads(raw)
    print("=== STRESS TEST RESULTS ===")
    all_correct = True
    for result in stress_results:
        status = "PASS" if result.get('correct_hierarchy', False) else "NEEDS WORK"
        if not result.get('correct_hierarchy', False):
            all_correct = False
        print(f"\n  {result['file']}: {status}")
        print(f"    Levels loaded: {result['loaded_levels']}")
        print(f"    Key instructions: {result['key_instructions']}")
    
    if all_correct:
        print("\nAll scenarios covered -- your hierarchy handles diverse file types correctly.")
    else:
        print("\nSome files lack appropriate guidance. Consider adding directory CLAUDE.md files.")
except json.JSONDecodeError:
    print(f"Failed to parse response:\n{raw}")

### Key Takeaways

1. **User-level (~/.claude/CLAUDE.md) is personal only.** Never put team rules here -- teammates won't see them.
2. **Project-level is the team baseline.** Universal standards go in .claude/CLAUDE.md or root CLAUDE.md, committed to git.
3. **Directory-level adds scoped context.** Backend, frontend, and infrastructure each get their own conventions.
4. **@import keeps things modular.** Split large configurations into focused files and import them.
5. **All levels merge.** They supplement each other, not replace -- avoid contradictions across levels.